# AI-Powered Data Extraction for Systematic Reviews with GPT-5

This notebook implements automated data extraction from research articles (abstracts or full texts) using OpenAI's GPT-5 reasoning models with structured outputs.

**Features:**
- Uses GPT-5 reasoning models with adjustable reasoning effort
- Extracts key systematic review data elements using structured outputs
- Works with abstracts or full-text articles
- Uses Pydantic models for guaranteed data structure compliance
- Extracts: population, location, exposure, outcome, study design, statistical methods, effect estimates
- Implements robust error handling and rate limiting
- Saves progress incrementally
- Generates extraction summary reports

**Extracted Data Elements:**
- **Population**: Study population characteristics (age, sex, health status, etc.)
- **Geographic Location**: City, province/state, country, region
- **Exposure**: Heat metrics (temperature, humidity, heat index, etc.)
- **Outcome**: Health outcomes (mortality, morbidity, hospitalizations, etc.)
- **Study Design**: Epidemiological study type
- **Statistical Methods**: Analytical approaches used
- **Effect Estimates**: Results with confidence intervals
- **Study Period**: Temporal coverage
- **Sample Size**: Number of participants/events

**Key GPT-5 Capabilities:**
- **Reasoning Effort**: Control depth of analysis (minimal, low, medium, high)
- **Structured Outputs**: Guaranteed Pydantic model compliance
- **Developer Messages**: Enhanced prompt control
- **No Temperature**: Uses reasoning_effort instead for consistency

**Requirements:**
- OpenAI SDK >= 1.0.0
- GPT-5, GPT-5-mini, or GPT-5-nano models

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import logging
import time
import os
from datetime import datetime
from typing import List, Optional, Literal
from tqdm import tqdm
from pydantic import BaseModel, Field
from openai import OpenAI, APIError, RateLimitError, APIConnectionError, APITimeoutError
from secret_keys import OPENAI_API_KEY

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("ai_extraction.log"),
        logging.StreamHandler()
    ]
)

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

logging.info("Setup complete - OpenAI client initialized")

## 2. Define Structured Output Schema using Pydantic

Comprehensive data extraction model for systematic reviews with all key data elements.

In [ ]:
class GeographicLocation(BaseModel):
    """Geographic location information."""
    city: Optional[str] = Field(default=None, description="City or cities where study was conducted")
    province_state: Optional[str] = Field(default=None, description="Province, state, or administrative region")
    country: Optional[str] = Field(default=None, description="Country where study was conducted")
    region: Optional[str] = Field(default=None, description="Broader geographic region (e.g., 'East Asia', 'Mediterranean', 'North America')")
    multi_location: bool = Field(default=False, description="Whether study includes multiple locations")


class ExposureMetric(BaseModel):
    """Heat exposure metric details."""
    metric_type: str = Field(description="Type of exposure metric (e.g., 'temperature', 'heat_index', 'wet_bulb_globe_temperature')")
    specific_measure: Optional[str] = Field(default=None, description="Specific measure (e.g., 'mean temperature', 'maximum temperature', 'minimum temperature')")
    unit: Optional[str] = Field(default=None, description="Unit of measurement (e.g., '°C', '°F', 'percentile')")
    lag_structure: Optional[str] = Field(default=None, description="Lag structure used (e.g., 'lag 0-3 days', 'distributed lag', 'cumulative')")


class OutcomeMeasure(BaseModel):
    """Health outcome measure details."""
    outcome_type: str = Field(description="Type of outcome (e.g., 'mortality', 'morbidity', 'hospitalization', 'emergency_department_visit')")
    specific_outcome: Optional[str] = Field(default=None, description="Specific outcome (e.g., 'all-cause mortality', 'cardiovascular mortality', 'heat stroke')")
    age_group: Optional[str] = Field(default=None, description="Age group if outcome is age-specific")


class EffectEstimate(BaseModel):
    """Effect estimate with confidence interval."""
    estimate_type: str = Field(description="Type of estimate (e.g., 'relative_risk', 'odds_ratio', 'hazard_ratio', 'percent_change', 'excess_deaths')")
    value: str = Field(description="Effect estimate value as string (preserve original format)")
    confidence_interval: Optional[str] = Field(default=None, description="Confidence interval as string (e.g., '95% CI: 1.05-1.15')")
    p_value: Optional[str] = Field(default=None, description="P-value if reported")
    comparison: Optional[str] = Field(default=None, description="What is being compared (e.g., 'per 1°C increase', 'extreme heat vs moderate')")
    subgroup: Optional[str] = Field(default=None, description="Subgroup if stratified analysis (e.g., 'elderly', 'outdoor workers')")


class DataExtraction(BaseModel):
    """
    Comprehensive structured data extraction for systematic reviews.
    
    Extracts all key data elements needed for evidence synthesis in
    systematic reviews of heat-health relationships.
    """
    
    # Population characteristics
    population_description: str = Field(
        description="Brief description of study population (e.g., 'general population', 'elderly aged 65+', 'outdoor workers')"
    )
    age_range: Optional[str] = Field(
        default=None,
        description="Age range of study population (e.g., '65-85 years', 'all ages')"
    )
    sex_distribution: Optional[str] = Field(
        default=None,
        description="Sex/gender distribution (e.g., '52% female', 'male only', 'both sexes')"
    )
    vulnerable_groups: List[str] = Field(
        default_factory=list,
        description="Vulnerable or special populations studied (e.g., 'elderly', 'children', 'chronic_disease', 'low_income')"
    )
    
    # Geographic location
    location: GeographicLocation = Field(
        description="Geographic location where study was conducted"
    )
    
    # Temporal information
    study_period_start: Optional[str] = Field(
        default=None,
        description="Start date/year of study period (e.g., '2000', 'June 2010')"
    )
    study_period_end: Optional[str] = Field(
        default=None,
        description="End date/year of study period (e.g., '2015', 'August 2020')"
    )
    season: Optional[str] = Field(
        default=None,
        description="Season(s) studied (e.g., 'summer months', 'May-September', 'warm season')"
    )
    
    # Exposure information
    exposures: List[ExposureMetric] = Field(
        description="List of heat exposure metrics examined"
    )
    exposure_source: Optional[str] = Field(
        default=None,
        description="Source of exposure data (e.g., 'meteorological stations', 'satellite data', 'reanalysis')"
    )
    
    # Outcome information
    outcomes: List[OutcomeMeasure] = Field(
        description="List of health outcomes examined"
    )
    outcome_source: Optional[str] = Field(
        default=None,
        description="Source of outcome data (e.g., 'death certificates', 'hospital records', 'national registry')"
    )
    
    # Sample size
    sample_size: Optional[str] = Field(
        default=None,
        description="Sample size or number of events (e.g., '50,000 participants', '12,453 deaths', '1,234 person-years')"
    )
    
    # Study design
    study_design: str = Field(
        description="Epidemiological study design (e.g., 'time_series', 'case_crossover', 'cohort', 'case_control', 'ecological')"
    )
    study_design_details: Optional[str] = Field(
        default=None,
        description="Additional design details (e.g., 'bidirectional case-crossover', 'multi-city time series')"
    )
    
    # Statistical methods
    statistical_methods: List[str] = Field(
        description="Statistical methods used (e.g., 'distributed_lag_non_linear_model', 'conditional_logistic_regression', 'cox_regression', 'poisson_regression', 'generalized_additive_model')"
    )
    confounders_adjusted: List[str] = Field(
        default_factory=list,
        description="Confounders adjusted for (e.g., 'air_pollution', 'day_of_week', 'long_term_trend', 'humidity')"
    )
    
    # Effect estimates
    effect_estimates: List[EffectEstimate] = Field(
        description="Main effect estimates reported in the study"
    )
    
    # Key findings
    key_finding_summary: str = Field(
        description="Brief summary of main findings (2-3 sentences)"
    )
    
    # Data quality indicators
    data_completeness: Optional[Literal["complete", "partial", "limited"]] = Field(
        default=None,
        description="How complete was the data extraction from available text"
    )
    extraction_confidence: Literal["high", "medium", "low"] = Field(
        description="Confidence in the accuracy of extracted data"
    )
    notes: Optional[str] = Field(
        default=None,
        description="Any important notes, limitations, or clarifications about the extraction"
    )

logging.info("Pydantic schema defined for data extraction")

## 3. AI Data Extraction Function

Extracts structured data from abstract or full text using OpenAI's API with structured outputs.

In [ ]:
def extract_data_with_ai(
    title: str,
    abstract: str,
    full_text: Optional[str] = None,
    model: str = "gpt-5",
    reasoning_effort: str = "medium",
    max_retries: int = 3
) -> dict:
    """
    Extract structured data from research article using OpenAI API.
    
    Uses GPT-5 with structured outputs (Pydantic) for guaranteed data structure.
    Prioritizes full text over abstract when available.
    GPT-5 uses reasoning_effort instead of temperature.
    
    Args:
        title: Article title
        abstract: Article abstract
        full_text: Full article text (optional, preferred over abstract)
        model: OpenAI model to use (gpt-5, gpt-5-mini, gpt-5-nano)
        reasoning_effort: Reasoning depth (minimal, low, medium, high)
        max_retries: Maximum number of retry attempts
    
    Returns:
        Dictionary with extracted data and metadata
    """
    
    # Determine which text to use
    if full_text and full_text.strip():
        article_text = full_text
        text_source = "full_text"
    else:
        article_text = abstract
        text_source = "abstract"
    
    # Construct the prompt
    # GPT-5 supports "developer" messages (functionally same as "system")
    developer_prompt = """You are an expert systematic reviewer and epidemiologist specializing in environmental health research.

Your task is to carefully extract structured data from research articles about heat exposure and health outcomes.

EXTRACTION GUIDELINES:

1. **Be precise and accurate**: Only extract information explicitly stated in the text
2. **Preserve original wording**: When extracting effect estimates, keep the original format and wording
3. **List all relevant items**: For exposures, outcomes, and statistical methods, extract all mentioned
4. **Geographic specificity**: Extract city, province/state, country, and region when available
5. **Population details**: Note vulnerable groups, age ranges, and special populations
6. **Effect estimates**: Extract all main results with confidence intervals
7. **Confounders**: List all variables adjusted for in the analysis
8. **Data quality**: Note if information is missing or unclear

COMMON EXPOSURE METRICS:
- Temperature (mean, maximum, minimum, apparent temperature)
- Heat index
- Wet bulb globe temperature (WBGT)
- Humidex
- Universal thermal climate index (UTCI)
- Heat waves (various definitions)

COMMON STUDY DESIGNS:
- Time series analysis
- Case-crossover design
- Cohort study
- Case-control study
- Ecological study

COMMON STATISTICAL METHODS:
- Distributed lag non-linear model (DLNM)
- Generalized additive model (GAM)
- Conditional logistic regression
- Cox proportional hazards
- Poisson regression
- Negative binomial regression

If information is not available in the text, use null/empty values. Do not infer or make assumptions.
"""
    
    user_prompt = f"""Title: {title}

{text_source.upper()}:
{article_text}

Please extract all relevant data from this article using the structured format."""
    
    for attempt in range(max_retries):
        try:
            # Use the beta.chat.completions.parse method for structured outputs
            # GPT-5 models support structured outputs with Pydantic
            completion = client.beta.chat.completions.parse(
                model=model,
                messages=[
                    {"role": "developer", "content": developer_prompt},  # GPT-5 uses "developer" role
                    {"role": "user", "content": user_prompt}
                ],
                response_format=DataExtraction,  # Pydantic model for structured output
                reasoning_effort=reasoning_effort,  # GPT-5 uses reasoning_effort instead of temperature
                # Note: temperature is deprecated for GPT-5 models
            )
            
            # Extract the parsed Pydantic object
            extraction_result = completion.choices[0].message.parsed
            
            # Convert to dictionary and add metadata
            result = extraction_result.model_dump()
            result['_metadata'] = {
                'text_source': text_source,
                'model': model,
                'reasoning_effort': reasoning_effort,
                'extraction_timestamp': datetime.now().isoformat()
            }
            
            return result
            
        except RateLimitError as e:
            wait = 2 ** attempt
            logging.warning(f"Rate limit exceeded. Retrying in {wait} seconds... (Attempt {attempt + 1}/{max_retries})")
            logging.warning(f"Error details: {e}")
            time.sleep(wait)
            
        except APIConnectionError as e:
            wait = 2 ** attempt
            logging.warning(f"Connection error. Retrying in {wait} seconds... (Attempt {attempt + 1}/{max_retries})")
            logging.warning(f"Error details: {e}")
            time.sleep(wait)
            
        except APITimeoutError as e:
            wait = 2 ** attempt
            logging.warning(f"Request timeout. Retrying in {wait} seconds... (Attempt {attempt + 1}/{max_retries})")
            logging.warning(f"Error details: {e}")
            time.sleep(wait)
            
        except APIError as e:
            logging.error(f"OpenAI API error: {e}")
            # For client errors (4xx), don't retry
            if hasattr(e, 'status_code') and 400 <= e.status_code < 500:
                logging.error("Client error - not retrying.")
                break
            # For server errors (5xx), retry
            if hasattr(e, 'status_code') and e.status_code >= 500:
                wait = 2 ** attempt
                logging.warning(f"Server error. Retrying in {wait} seconds... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
            else:
                break
                
        except Exception as e:
            logging.error(f"Unexpected error: {type(e).__name__}: {e}")
            break
    
    # Fallback if all retries failed
    logging.error("Failed to extract data after multiple attempts")
    return {
        "population_description": "Extraction failed",
        "location": {
            "city": None,
            "province_state": None,
            "country": None,
            "region": None,
            "multi_location": False
        },
        "exposures": [],
        "outcomes": [],
        "study_design": "Unknown",
        "statistical_methods": [],
        "effect_estimates": [],
        "key_finding_summary": "Failed to extract data due to API error",
        "extraction_confidence": "low",
        "notes": "Extraction failed after multiple retry attempts",
        "_metadata": {
            "text_source": text_source,
            "model": model,
            "reasoning_effort": reasoning_effort,
            "extraction_timestamp": datetime.now().isoformat(),
            "error": "API call failed"
        }
    }

logging.info("AI extraction function defined with structured outputs for GPT-5")

## 4. Load Data for Extraction

Load the screened articles that were marked for inclusion.

In [ ]:
def load_articles_for_extraction(
    screening_file: str = "outputs/ai_screening_full_results.csv",
    include_uncertain: bool = False
) -> pd.DataFrame:
    """
    Load articles that need data extraction.
    
    Args:
        screening_file: Path to screening results CSV
        include_uncertain: Whether to include articles marked as 'uncertain'
    
    Returns:
        DataFrame with articles for extraction
    """
    try:
        df = pd.read_csv(screening_file)
        
        # Filter to included articles
        if include_uncertain:
            df_extract = df[df['ai_decision'].isin(['include', 'uncertain'])].copy()
        else:
            df_extract = df[df['ai_decision'] == 'include'].copy()
        
        logging.info(f"Loaded {len(df_extract)} articles for extraction from {screening_file}")
        
        # Ensure required columns
        required_cols = ['PMID', 'Title', 'Abstract']
        missing_cols = [col for col in required_cols if col not in df_extract.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        # Clean data
        df_extract['Abstract'] = df_extract['Abstract'].fillna('')
        df_extract['Title'] = df_extract['Title'].fillna('')
        
        # Add FullText column if not present (for future use)
        if 'FullText' not in df_extract.columns:
            df_extract['FullText'] = None
        
        return df_extract
        
    except FileNotFoundError:
        logging.error(f"Screening file not found: {screening_file}")
        logging.info("Please run the screening notebook first.")
        raise
    except Exception as e:
        logging.error(f"Error loading articles: {e}")
        raise

# Load articles
# Uncomment and modify the path as needed:
# df_articles = load_articles_for_extraction("outputs/ai_screening_full_results.csv")
# print(f"Loaded {len(df_articles)} articles for data extraction")
# df_articles.head()

## 5. Test Single Article Extraction

In [ ]:
# Test extraction on a single article
# Uncomment to run after loading data:

# test_idx = 0
# test_row = df_articles.iloc[test_idx]

# print(f"Testing extraction on PMID: {test_row['PMID']}")
# print(f"\nTitle: {test_row['Title']}")
# print(f"\nAbstract (first 300 chars): {test_row['Abstract'][:300]}...")

# print("\n" + "="*80)
# print("Running AI data extraction with GPT-5 (medium reasoning effort)...")
# print("="*80)

# test_extraction = extract_data_with_ai(
#     title=test_row['Title'],
#     abstract=test_row['Abstract'],
#     full_text=test_row.get('FullText'),
#     model="gpt-5",
#     reasoning_effort="medium"
# )

# print("\n" + "="*80)
# print("EXTRACTION RESULT:")
# print("="*80)
# print(json.dumps(test_extraction, indent=2, default=str))

## 6. Batch Data Extraction Function

In [ ]:
def batch_extract_data(
    df: pd.DataFrame,
    output_file: str = "outputs/ai_extraction_results.csv",
    batch_size: int = 20,
    rate_limit_delay: float = 1.0,
    model: str = "gpt-5",
    reasoning_effort: str = "medium"
) -> pd.DataFrame:
    """
    Extract data from all articles in batches with progress saving.
    
    Args:
        df: DataFrame with articles to extract
        output_file: Path to save extraction results
        batch_size: Save progress every N records
        rate_limit_delay: Seconds to wait between API calls
        model: GPT-5 model to use (gpt-5, gpt-5-mini, gpt-5-nano)
        reasoning_effort: Reasoning depth (minimal, low, medium, high)
    
    Returns:
        DataFrame with extraction results
    """
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    # Check if output file exists to resume
    if os.path.exists(output_file):
        df_existing = pd.read_csv(output_file)
        processed_pmids = set(df_existing['PMID'].astype(str))
        logging.info(f"Found {len(processed_pmids)} already processed records")
    else:
        processed_pmids = set()
        df_existing = None
    
    # Filter to unprocessed records
    df['PMID'] = df['PMID'].astype(str)
    df_to_process = df[~df['PMID'].isin(processed_pmids)].copy()
    
    logging.info(f"Processing {len(df_to_process)} records with {model} (reasoning_effort={reasoning_effort})")
    
    results = []
    
    # Process with progress bar
    for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Extracting data"):
        
        # Extract data
        extraction = extract_data_with_ai(
            title=row['Title'],
            abstract=row['Abstract'],
            full_text=row.get('FullText'),
            model=model,
            reasoning_effort=reasoning_effort
        )
        
        # Flatten nested structures for CSV
        result_row = {
            'PMID': row['PMID'],
            'Title': row['Title'],
            'Abstract': row['Abstract'],
            'Authors': row.get('Authors', ''),
            'Date': row.get('Date', ''),
            'Journal': row.get('Journal', ''),
            
            # Population
            'population_description': extraction['population_description'],
            'age_range': extraction.get('age_range'),
            'sex_distribution': extraction.get('sex_distribution'),
            'vulnerable_groups': json.dumps(extraction.get('vulnerable_groups', [])),
            
            # Location
            'city': extraction['location'].get('city'),
            'province_state': extraction['location'].get('province_state'),
            'country': extraction['location'].get('country'),
            'region': extraction['location'].get('region'),
            'multi_location': extraction['location'].get('multi_location', False),
            
            # Temporal
            'study_period_start': extraction.get('study_period_start'),
            'study_period_end': extraction.get('study_period_end'),
            'season': extraction.get('season'),
            
            # Exposure
            'exposures': json.dumps(extraction['exposures']),
            'exposure_source': extraction.get('exposure_source'),
            
            # Outcome
            'outcomes': json.dumps(extraction['outcomes']),
            'outcome_source': extraction.get('outcome_source'),
            
            # Sample
            'sample_size': extraction.get('sample_size'),
            
            # Study design
            'study_design': extraction['study_design'],
            'study_design_details': extraction.get('study_design_details'),
            
            # Statistics
            'statistical_methods': json.dumps(extraction['statistical_methods']),
            'confounders_adjusted': json.dumps(extraction.get('confounders_adjusted', [])),
            
            # Results
            'effect_estimates': json.dumps(extraction['effect_estimates']),
            'key_finding_summary': extraction['key_finding_summary'],
            
            # Quality
            'data_completeness': extraction.get('data_completeness'),
            'extraction_confidence': extraction['extraction_confidence'],
            'notes': extraction.get('notes'),
            
            # Metadata
            'text_source': extraction['_metadata']['text_source'],
            'extraction_model': extraction['_metadata']['model'],
            'extraction_reasoning_effort': extraction['_metadata']['reasoning_effort'],
            'extraction_timestamp': extraction['_metadata']['extraction_timestamp']
        }
        
        results.append(result_row)
        
        # Save progress every batch_size records
        if len(results) % batch_size == 0:
            df_results = pd.DataFrame(results)
            
            # Append to existing or create new
            if df_existing is not None:
                df_combined = pd.concat([df_existing, df_results], ignore_index=True)
            else:
                df_combined = df_results
            
            df_combined.to_csv(output_file, index=False)
            logging.info(f"Saved progress: {len(df_combined)} total records")
            
            # Update for next batch
            df_existing = df_combined
            results = []
        
        # Rate limiting
        time.sleep(rate_limit_delay)
    
    # Save any remaining results
    if results:
        df_results = pd.DataFrame(results)
        
        if df_existing is not None:
            df_combined = pd.concat([df_existing, df_results], ignore_index=True)
        else:
            df_combined = df_results
        
        df_combined.to_csv(output_file, index=False)
        logging.info(f"Final save: {len(df_combined)} total records")
    else:
        df_combined = df_existing if df_existing is not None else pd.DataFrame()
    
    return df_combined

logging.info("Batch extraction function defined")

## 7. Run Extraction on Test Subset

In [ ]:
# Test on first 5 records
# Uncomment to run:

# df_test = df_articles.head(5).copy()

# print(f"Testing extraction on {len(df_test)} records with GPT-5...")
# print(f"Model: gpt-5 | Reasoning effort: medium")
# print(f"Estimated cost: ~$0.10-0.30 (varies with reasoning tokens)")
# print("="*80)

# df_test_results = batch_extract_data(
#     df=df_test,
#     output_file="outputs/ai_extraction_test_results.csv",
#     batch_size=3,
#     rate_limit_delay=0.5,
#     model="gpt-5",
#     reasoning_effort="medium"
# )

# print("\n" + "="*80)
# print("✓ Test extraction complete!")
# print("="*80)
# print(f"Results saved to: outputs/ai_extraction_test_results.csv")
# print(f"\nExtracted data from {len(df_test_results)} articles")
# df_test_results.head()

## 8. Run Full Extraction

In [ ]:
# UNCOMMENT TO RUN FULL EXTRACTION
# WARNING: This will make many API calls and may be expensive

# print(f"Starting full extraction of {len(df_articles)} articles...")
# print("This may take a long time and cost money. Press Ctrl+C to cancel.")

# df_full_results = batch_extract_data(
#     df=df_articles,
#     output_file="outputs/ai_extraction_full_results.csv",
#     batch_size=20,
#     rate_limit_delay=1.0,
#     model="gpt-5",  # Use gpt-5, gpt-5-mini, or gpt-5-nano
#     reasoning_effort="medium"  # minimal, low, medium, or high
# )

# print("\nFull extraction complete!")
# print(f"Results saved to: outputs/ai_extraction_full_results.csv")

## 9. Analyze Extraction Results

In [ ]:
def analyze_extraction_results(results_file: str = "outputs/ai_extraction_full_results.csv"):
    """
    Generate summary statistics from extraction results.
    """
    df = pd.read_csv(results_file)
    
    print("="*80)
    print("DATA EXTRACTION SUMMARY")
    print("="*80)
    print(f"\nTotal articles extracted: {len(df)}")
    
    print("\nExtraction Quality:")
    print(df['extraction_confidence'].value_counts())
    
    print("\nData Completeness:")
    print(df['data_completeness'].value_counts())
    
    print("\nText Source:")
    print(df['text_source'].value_counts())
    
    print("\nStudy Designs:")
    print(df['study_design'].value_counts())
    
    print("\nGeographic Distribution:")
    print(f"Countries represented: {df['country'].nunique()}")
    print("\nTop 10 countries:")
    print(df['country'].value_counts().head(10))
    
    print("\nSample Size Information:")
    print(f"Articles with sample size: {df['sample_size'].notna().sum()} ({df['sample_size'].notna().sum()/len(df)*100:.1f}%)")
    
    # Parse JSON fields for more detailed analysis
    print("\nCommon Exposure Metrics:")
    all_exposures = []
    for exposures_str in df['exposures'].dropna():
        try:
            exposures = json.loads(exposures_str)
            for exp in exposures:
                all_exposures.append(exp.get('metric_type', 'unknown'))
        except:
            pass
    if all_exposures:
        exposure_counts = pd.Series(all_exposures).value_counts()
        print(exposure_counts.head(10))
    
    print("\nCommon Statistical Methods:")
    all_methods = []
    for methods_str in df['statistical_methods'].dropna():
        try:
            methods = json.loads(methods_str)
            all_methods.extend(methods)
        except:
            pass
    if all_methods:
        method_counts = pd.Series(all_methods).value_counts()
        print(method_counts.head(10))
    
    return df

# Analyze results
# Uncomment to run:
# if os.path.exists("outputs/ai_extraction_full_results.csv"):
#     df_analysis = analyze_extraction_results("outputs/ai_extraction_full_results.csv")
# else:
#     print("No results file found. Run extraction first.")

## 10. Export Structured Data for Analysis

In [ ]:
def export_structured_data(
    results_file: str = "outputs/ai_extraction_full_results.csv",
    output_dir: str = "outputs/structured_data"
):
    """
    Export extracted data in various structured formats for analysis.
    """
    df = pd.read_csv(results_file)
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. High-quality extractions only
    df_high_quality = df[df['extraction_confidence'].isin(['high', 'medium'])].copy()
    df_high_quality.to_csv(f"{output_dir}/high_quality_extractions.csv", index=False)
    print(f"Exported {len(df_high_quality)} high-quality extractions")
    
    # 2. Study characteristics table
    study_chars = df[[
        'PMID', 'Authors', 'Date', 'Title',
        'country', 'city', 'study_period_start', 'study_period_end',
        'study_design', 'sample_size', 'population_description'
    ]].copy()
    study_chars.to_csv(f"{output_dir}/study_characteristics.csv", index=False)
    print(f"Exported study characteristics table")
    
    # 3. Effect estimates table (flattened)
    effect_rows = []
    for idx, row in df.iterrows():
        try:
            effects = json.loads(row['effect_estimates'])
            for effect in effects:
                effect_rows.append({
                    'PMID': row['PMID'],
                    'Authors': row['Authors'],
                    'country': row['country'],
                    'estimate_type': effect.get('estimate_type'),
                    'value': effect.get('value'),
                    'confidence_interval': effect.get('confidence_interval'),
                    'p_value': effect.get('p_value'),
                    'comparison': effect.get('comparison'),
                    'subgroup': effect.get('subgroup')
                })
        except:
            pass
    
    df_effects = pd.DataFrame(effect_rows)
    df_effects.to_csv(f"{output_dir}/effect_estimates.csv", index=False)
    print(f"Exported {len(df_effects)} effect estimates")
    
    # 4. Full data as JSON (preserves nested structures)
    df.to_json(f"{output_dir}/full_extraction_data.json", orient='records', indent=2)
    print(f"Exported full data as JSON")
    
    # 5. Summary statistics
    summary = {
        'total_articles': len(df),
        'countries': df['country'].nunique(),
        'study_designs': df['study_design'].value_counts().to_dict(),
        'extraction_quality': df['extraction_confidence'].value_counts().to_dict(),
        'extraction_date': datetime.now().isoformat()
    }
    
    with open(f"{output_dir}/extraction_summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Exported extraction summary")
    
    print(f"\nAll files saved to: {output_dir}/")

# Export data
# Uncomment to run:
# if os.path.exists("outputs/ai_extraction_full_results.csv"):
#     export_structured_data("outputs/ai_extraction_full_results.csv")
# else:
#     print("No results file found. Run extraction first.")

## 11. Summary and Next Steps

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════════════════
AI-POWERED DATA EXTRACTION FOR SYSTEMATIC REVIEWS - SUMMARY (GPT-5)
═══════════════════════════════════════════════════════════════════════════════

✓ Latest OpenAI SDK (>=1.0.0) with native structured outputs
✓ GPT-5 reasoning models with adjustable reasoning_effort
✓ Pydantic BaseModel for type-safe, validated responses
✓ Comprehensive extraction of systematic review data elements
✓ Works with abstracts or full-text articles
✓ Robust error handling and retry logic
✓ Incremental progress saving with resume capability
✓ Multiple export formats for downstream analysis

EXTRACTED DATA ELEMENTS:

1. Population & Demographics
   - Population description
   - Age range and sex distribution
   - Vulnerable groups
   - Sample size

2. Geographic Information
   - City, province/state, country, region
   - Multi-location indicator

3. Temporal Information
   - Study period (start/end)
   - Season studied

4. Exposure Assessment
   - Exposure metrics (temperature, heat index, WBGT, etc.)
   - Units and lag structures
   - Data sources

5. Health Outcomes
   - Outcome types and specific outcomes
   - Age-specific outcomes
   - ICD codes
   - Data sources

6. Study Design & Methods
   - Study design type
   - Statistical methods
   - Confounders adjusted

7. Results
   - Effect estimates with CIs
   - Subgroup analyses
   - Key findings summary

8. Quality Indicators
   - Data completeness
   - Extraction confidence
   - Notes and limitations

WORKFLOW:

1. LOAD DATA:
   - Import articles from screening results
   - Filter to included studies
   - Check for full-text availability

2. TEST EXTRACTION:
   - Run on 5-10 sample articles
   - Review extraction quality
   - Adjust prompts if needed

3. VALIDATE:
   - Compare with manual extraction on subset
   - Calculate agreement metrics
   - Refine extraction criteria

4. FULL EXTRACTION:
   - Process all included articles
   - Monitor costs and progress
   - Handle errors and retries

5. QUALITY ASSURANCE:
   - Review low-confidence extractions
   - Validate a random sample (10-20%)
   - Manually extract missing data

6. EXPORT & ANALYZE:
   - Export structured data tables
   - Create summary statistics
   - Prepare for meta-analysis

GPT-5 MODEL RECOMMENDATIONS:
- Development/Testing: gpt-5-mini with medium reasoning (good balance)
- Production: gpt-5 with medium reasoning (best accuracy)
- Complex Articles: gpt-5 with high reasoning (deepest analysis)
- Bulk Extraction: gpt-5-nano with low/minimal reasoning (cost-effective)

REASONING EFFORT GUIDE:
- minimal: Fastest, cheapest, shallow reasoning (simple extraction)
- low: Fast, light reasoning (straightforward articles)
- medium: Balanced depth vs. speed (recommended default)
- high: Deep multistep reasoning (complex, detailed articles)

ESTIMATED COSTS (per 100 articles with abstracts ~300 words):
- gpt-5-nano (minimal): ~$5-10
- gpt-5-mini (medium): ~$15-30
- gpt-5 (medium): ~$30-60
- gpt-5 (high): ~$60-120
Note: Costs vary based on reasoning tokens used by the model

KEY GPT-5 FEATURES:
- Reasoning effort controls depth of analysis
- Developer messages (instead of system messages)
- Temperature parameter is deprecated
- Supports structured outputs with Pydantic
- Higher accuracy on complex extraction tasks
- Transparent reasoning process

TIPS FOR SUCCESS:

1. Use full text when available for better extraction quality
2. Start with medium reasoning effort, adjust based on results
3. Use high reasoning effort for complex or ambiguous articles
4. Validate extraction on a subset before full run
5. Review low-confidence extractions manually
6. Use gpt-5 for detailed articles, gpt-5-mini for straightforward ones
7. Save progress frequently (batch_size=20)
8. Export to multiple formats for different analyses
9. Document extraction decisions and validation process
10. Consider dual extraction for critical data elements
11. Monitor reasoning tokens usage to optimize costs

NEXT STEPS:

1. Load screening results and articles for extraction
2. Test on sample articles
3. Validate extraction quality
4. Run full extraction
5. Export structured data
6. Proceed to meta-analysis or evidence synthesis

═══════════════════════════════════════════════════════════════════════════════
""")